# Notebook to extract data from 69

This notebook gather code exemple and exploration for a complete data cleaning pipeline and extraction used in the web application. 

In [1]:
import pandas as pd
import datetime as dt

from helpers.api_requesters import Ademe_API_requester

In [101]:
# Instanciation
# ademe_requester = Ademe_API_requester()

In [102]:
# data = ademe_requester.get_bydepartement(69, neuf=True)

# df = pd.DataFrame(data)

# df.to_csv('output/ademe_data_69_neuf.csv', index=False)

# Concatenation and data cleaning

In [103]:
relevant_columns = [
    "cout_total_5_usages",
    "cout_chauffage",
    "cout_eclairage",
    "cout_refroidissement",
    "cout_auxiliaires",
    "cout_ecs",
    "conso_5_usages_ef",
    "conso_chauffage_ef",
    "conso_eclairage_ef",
    "conso_auxiliaires_ef",
    "conso_ecs_ef",
    "conso_refroidissement_ef",
    "surface_habitable_logement",
    "nombre_niveau_logement",
    "type_batiment",
    "annee_construction",
    "code_insee_ban",
    "code_departement_ban",
    "etiquette_dpe",
    "etiquette_ges",
    "nom_commune_ban",
    "code_postal_ban",
    "emission_ges_chauffage",
    "emission_ges_eclairage",
    "emission_ges_ecs",
    "emission_ges_5_usages",
    "emission_ges_auxiliaires",
    "emission_ges_refroidissement",
    "_geopoint",
    "type_energie_principale_chauffage",
    "date_reception_dpe",
    "numero_dpe",
]

In [104]:
len(relevant_columns)

32

In [105]:
df_old = pd.read_csv('output/ademe_data_69.csv', low_memory=False, dtype={"numero_dpe": str})
df_new = pd.read_csv('output/ademe_data_69_neuf.csv', low_memory=False, dtype={"numero_dpe": str})

# add the year to the new dataframe
df_new["annee_construction"] = dt.datetime.now().year

In [106]:
# Concatenate old and new dataframes and drop duplicates
df = pd.concat([df_old, df_new]).drop_duplicates().reset_index(drop=True)

In [107]:
df = df[relevant_columns]

## Année

In [108]:
df = df.dropna(subset=["annee_construction"])

# Replace by age of building.
current_year = dt.datetime.now().year
df.loc[:, "age_batiment"] = current_year - df.loc[:, "annee_construction"]

# Delete all negative values.
df = df.loc[df["age_batiment"] >= 0, :]

In [109]:
len(df)

284991

## Cout 5 usages

In [110]:
df = df.dropna(subset=["cout_total_5_usages"])

# Suppression des lignes aberrantes (méthode de l'inter-quartile).
Q1 = df["cout_total_5_usages"].quantile(0.25)
Q3 = df["cout_total_5_usages"].quantile(0.75)
IQR = Q3 - Q1

# Filtering the dataframe
df = df[
    (df["cout_total_5_usages"] >= Q1 - 1.5 * IQR)
    & (df["cout_total_5_usages"] <= Q3 + 1.5 * IQR)
]

In [111]:
len(df)

267747

## Type_energie_principale_chauffage

In [112]:
# Distribution.
chauffage_distribution = (
    df["type_energie_principale_chauffage"].value_counts(normalize=True) * 100
)

threshold = 15

# Getting the list of all the modalities not reaching the threshold.
rare_modalities = chauffage_distribution[
    chauffage_distribution < threshold
].index.tolist()

# Replacing rare modalities by "Autre" in the dataframe.
df.loc[:, "type_energie_principale_chauffage"] = df[
    "type_energie_principale_chauffage"
].replace(rare_modalities, "Autre")

## surface_habitable_logement

In [113]:
# Using interquartile method to eliminate outliers (and missing values).
Q1 = df["surface_habitable_logement"].quantile(0.25)
Q3 = df["surface_habitable_logement"].quantile(0.75)
IQR = Q3 - Q1

# Filtering the dataframe
df = df[
    (df["surface_habitable_logement"] >= Q1 - 1.5 * IQR)
    & (df["surface_habitable_logement"] <= Q3 + 1.5 * IQR)
]

## Nombre niveau logement

In [114]:
# Discard missing values.
df.loc[:, "nombre_niveau_logement"].dropna(inplace=True)
# Discard outliers.
df = df[(df["nombre_niveau_logement"] >= 1) & (df["nombre_niveau_logement"] <= 10)]

## Code INSEE

In [115]:
# Changing code_insee for string.
df["code_insee_ban"] = df["code_insee_ban"].astype(str)

# Adding a 0 to the insee code being too short.
df["code_insee_ban"] = df["code_insee_ban"].apply(lambda x: x.zfill(5))

## Department

In [116]:
# Adding heading 0 for deparment codes being too short.
df["code_departement_ban"] = df["code_departement_ban"].astype(str)
df["code_departement_ban"] = df["code_departement_ban"].apply(lambda x: x.zfill(2))

## Adding lat and lon to the dataframe

In [117]:
df[["lat", "lon"]] = (df["_geopoint"].str.split(",", expand=True).astype(float))

df.drop(columns=["_geopoint"], inplace=True)

## Mapping climate area

In [118]:
H1 = [
    "01",
    "02",
    "03",
    "05",
    "08",
    "10",
    "14",
    "15",
    "19",
    "21",
    "23",
    "25",
    "27",
    "28",
    "38",
    "39",
    "42",
    "43",
    "45",
    "51",
    "52",
    "54",
    "55",
    "57",
    "58",
    "59",
    "60",
    "61",
    "62",
    "63",
    "67",
    "68",
    "69",
    "70",
    "71",
    "73",
    "74",
    "75",
    "76",
    "77",
    "78",
    "80",
    "87",
    "88",
    "89",
    "90",
    "91",
    "92",
    "93",
    "94",
    "95",
]

H2 = [
    "04",
    "07",
    "09",
    "12",
    "16",
    "17",
    "18",
    "22",
    "24",
    "26",
    "29",
    "31",
    "32",
    "33",
    "35",
    "36",
    "37",
    "40",
    "41",
    "44",
    "46",
    "47",
    "48",
    "49",
    "50",
    "53",
    "56",
    "64",
    "65",
    "72",
    "79",
    "81",
    "82",
    "84",
    "85",
    "86",
]

H3 = ["06", "11", "13", "20", "30", "34", "66", "83"]

In [119]:
# Codage des zones climatiques.
df.loc[df["code_departement_ban"].isin(H1), "zone_climatique"] = "H1"
df.loc[df["code_departement_ban"].isin(H2), "zone_climatique"] = "H2"
df.loc[df["code_departement_ban"].isin(H3), "zone_climatique"] = "H3"

## Altitude

In [120]:
# Get the altitude.
cities = pd.read_csv("data/communes-france-2025.csv", low_memory=False)

# Groupby department and take the mean altitude.
departments = cities.groupby("dep_code")["altitude_moyenne"].mean()

# Isolate the merging columns.
cities = cities[["code_insee", "altitude_moyenne"]]

In [121]:
# Use the insee code to determine the altitude.
df = df.merge(cities, left_on="code_insee_ban", right_on="code_insee", how="left")

In [122]:
# Completing missing altitude values using the Elevation API.
missing = df[df["altitude_moyenne"].isna()].drop(columns=["altitude_moyenne"])

# Completing the missing values using the mean altitude per department.
missing = missing.merge(
    departments, left_on="code_departement_ban", right_index=True, how="left"
)

df.loc[missing.index, "altitude_moyenne"] = missing["altitude_moyenne"]

In [123]:
len(df)

250924

# Date DPE data

In [124]:
df["date_reception_dpe"]

0         2022-06-03
1         2022-02-23
2         2022-03-24
3         2022-03-08
4         2022-04-12
             ...    
250919    2025-03-10
250920    2025-04-15
250921    2024-11-25
250922    2025-07-31
250923    2025-09-25
Name: date_reception_dpe, Length: 250924, dtype: object

In [ ]:
df.to_csv('output/ademe_data_69_cleaned.csv', index=False)